# Sketch-Assisted ACDC on official MIB IOI

**Stage A is the default execution path.** Stages B and C require explicit flags after the numerical checks pass. The 16 selected items are EAP-ranked sender-to-receiver **edges** in MIB's complete GPT-2 Small graph. Results here describe restricted sequential pruning, not a full ACDC circuit or an official MIB leaderboard score.

MIB graph/patching: `external/MIB-circuit-track/EAP-IG/src/eap/{graph,utils,evaluate,attribute}.py` at the commit recorded below. Original ACDC traversal/threshold: `external/Automatic-Circuit-Discovery/acdc/TLACDCExperiment.py`, `TLACDCCorrespondence.py`. MIB exact attribution toggles each edge independently, whereas ACDC updates its current graph after each accepted removal. This notebook implements that sequential behavior on a fixed candidate set. No upstream source is modified.

In [ ]:
from pathlib import Path
import os, sys, json, time, subprocess, importlib.metadata
from datetime import datetime, timezone
import numpy as np, pandas as pd, torch, matplotlib.pyplot as plt
ROOT = Path.cwd().resolve()
if not (ROOT / 'mib_ioi/runtime.py').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from mib_ioi.runtime import configure_repository, seed_everything, load_model, load_splits, Ledger, write_json, subset
from src.sketch_acdc.core import EdgeExperiment, select_edges, exact_acdc, hessian_vector_product, nystrom_sketch
SEED = 42
PILOT = os.environ.get('SKETCH_ACDC_PROFILE') == 'pilot'
TRAIN_N, VALIDATION_N = (16, 8) if PILOT else (32, 64)
BATCH_SIZE, EAP_BATCH_SIZE = 2, 2
EDGE_COUNT = 16
SMOKE_COUNT = TRAIN_N
RUN_STAGE_B = os.environ.get('SKETCH_ACDC_RUN_B') == '1'
RUN_STAGE_C = os.environ.get('SKETCH_ACDC_RUN_C') == '1'
os.environ.setdefault('HF_HOME', str(ROOT / '.cache/huggingface'))
os.environ.setdefault('MPLCONFIGDIR', str(ROOT / '.cache/matplotlib'))
os.environ.setdefault('HF_DATASETS_OFFLINE', '1')
os.environ.setdefault('HF_HUB_OFFLINE', '1')
OUT = ROOT / 'results/mib_sketch_acdc' / datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
OUT.mkdir(parents=True, exist_ok=True)
seed_everything(SEED)
print('Results:', OUT, 'batch size:', BATCH_SIZE, 'pilot:', PILOT, 'stage B/C:', RUN_STAGE_B, RUN_STAGE_C)

Seed: 42; TF32 disabled; model.eval() used.
Results: H:\GenAI Research\sketch_prune\results\mib_sketch_acdc\20261015T054348Z batch size: 2 pilot: True stage B/C: True True


## Stage A — official data, edge rankings, intervention and derivative checks

The IOI score is MIB's per-example correct-minus-incorrect logit difference. A removed edge adds its corrupted-minus-clean sender output to precisely its receiver input. The continuous coefficient multiplies that same update; its zero and one endpoints are checked against MIB's independent graph evaluator. All model parameters remain frozen after EAP attribution.

In [ ]:
repo, provenance = configure_repository(ROOT)
from eap.graph import Graph
from eap.attribute import get_scores_eap
from MIB_circuit_track.metrics import get_metric
acdc_dir = ROOT / 'external/Automatic-Circuit-Discovery'
acdc_commit = subprocess.check_output(['git','-C',str(acdc_dir),'rev-parse','HEAD'], text=True).strip()
model = load_model('cuda')
datasets, manifest, split_info = load_splits(model, {'train':TRAIN_N, 'validation':VALIDATION_N}, SEED)
ledger = Ledger('cuda', OUT)
manifest.to_csv(OUT/'example_manifest.csv', index=False)
print('MIB:', provenance['mib_commit'], 'EAP:', provenance['eap_commit'], 'ACDC:', acdc_commit)
print('Splits:', {k:len(v) for k,v in datasets.items()}, 'full split overlap:', split_info['full_filtered_split_clean_prompt_overlap'])
print(manifest[['split','clean_prompt','corrupted_prompt','correct_answer']].head(3).to_string(index=False))

Loaded pretrained model gpt2-small into HookedTransformer
Using the latest cached version of the dataset since mib-bench/ioi couldn't be found on the Hugging Face Hub (offline mode is enabled).
Found the latest cached dataset configuration 'default' at H:\GenAI Research\sketch_prune\.cache\huggingface\datasets\mib-bench___ioi\default\0.0.0\e5f3468f3af4c0883be35cd3bced8c711c95d286 (last modified on Wed Oct 14 18:13:25 2026).
Using the latest cached version of the dataset since mib-bench/ioi couldn't be found on the Hugging Face Hub (offline mode is enabled).
Found the latest cached dataset configuration 'default' at H:\GenAI Research\sketch_prune\.cache\huggingface\datasets\mib-bench___ioi\default\0.0.0\e5f3468f3af4c0883be35cd3bced8c711c95d286 (last modified on Wed Oct 14 18:13:25 2026).
MIB: b759df34433c9e31043ba9e02908ce0bf20e894f EAP: 7af394a5662de8b23ad6154716a0cd3993d447a3 ACDC: bc99ace817974b5584b7ee203d596a8e2bbcd399
Splits: {'train': 16, 'validation': 8} full split overlap: {'tr

In [ ]:
graph = Graph.from_model(model)
graph.reset(empty=False)
def metric_loss(logits, clean_logits, lengths, labels):
    value = get_metric('logit_diff','ioi',model.tokenizer,model)(logits, clean_logits, lengths, labels, mean=True, loss=True)
    def count_backward(gradient):
        ledger.backward_calls += 1
        return gradient
    value.register_hook(count_backward)
    return value
print('Official EAP ranking:', TRAIN_N, 'train examples,', (TRAIN_N+EAP_BATCH_SIZE-1)//EAP_BATCH_SIZE, 'batches')
with ledger.instrument(model), ledger.stage('official_edge_eap', estimate='3 forward batches plus one backward per EAP batch'):
    try:
        model.requires_grad_(True)
        with torch.enable_grad():
            scores = get_scores_eap(model, graph, datasets['train'].to_dataloader(EAP_BATCH_SIZE), metric_loss, quiet=True)
    finally:
        model.zero_grad(set_to_none=True)
        model.requires_grad_(False)
edges = select_edges(graph, scores, EDGE_COUNT)
selected = pd.DataFrame([vars(edge) for edge in edges])
selected.to_csv(OUT/'selected_edges.csv', index=False)
print(selected[['name','eap_score']].to_string(index=False))
identity = json.dumps({'model':'gpt2','task':'ioi','mib':provenance['mib_commit'],'eap':provenance['eap_commit'], 'seed':SEED, 'intervention':'patching'}, sort_keys=True)
experiment_splits = dict(datasets)
# First 16 training prompts estimate derivatives; the other 16 calibrate residuals.
experiment_splits['calibration'] = subset(datasets['train'], range(TRAIN_N//2,TRAIN_N))
experiment_splits['smoke'] = subset(datasets['train'], range(SMOKE_COUNT))
experiment = EdgeExperiment(model, graph, edges, experiment_splits, BATCH_SIZE, identity, ledger)
config = {'model':'gpt2','task':'ioi','dataset':'official MIB IOI','splits':split_info,
          'seed':SEED,'train_examples':TRAIN_N,'validation_examples':VALIDATION_N,
          'batch_size':BATCH_SIZE,'eap_batch_size':EAP_BATCH_SIZE,'selected_edges':selected.to_dict('records'),
          'intervention':'MIB corrupted-minus-clean sender activation at receiver input',
          'mib_provenance':provenance,'acdc_commit':acdc_commit,
          'profile':'pilot' if PILOT else 'full','run_stage_b':RUN_STAGE_B,'run_stage_c':RUN_STAGE_C,
          'sketch_ranks':[2,4] if PILOT else [2,4,8,16],
          'packages':{p:importlib.metadata.version(p) for p in ['torch','transformer-lens','datasets','numpy','pandas']}}
write_json(OUT/'experiment_config.json',config)

Official EAP ranking: 16 train examples, 8 batches
official_edge_eap: 3 forward batches plus one backward per EAP batch
official_edge_eap: 24 actual forward batches, 4.16s wall
             name  eap_score
    a9.h9->logits   2.233752
   a10.h7->logits  -1.429598
    a9.h6->logits   1.063488
  a5.h5->a8.h6<v>   0.970336
  a11.h10->logits  -0.722662
a9.h9->a11.h10<q>  -0.469756
 a6.h9->a11.h8<k>   0.000224
  a1.h2->a3.h6<k>  -0.000224
  a2.h1->a6.h3<k>  -0.000224
  a3.h8->a6.h1<v>  -0.000224
  a0.h2->a5.h5<q>  -0.000224
  a2.h7->a9.h0<v>  -0.000041
  a2.h4->a5.h8<k>  -0.000041
a3.h10->a6.h10<k>   0.000041
 a2.h10->a5.h8<k>  -0.000041
 a8.h4->a11.h0<k>   0.000041


In [ ]:
# Compare exact graph patching and differentiable interpolation endpoints on a tiny official subset.
from mib_ioi.runtime import subset
from eap.evaluate import evaluate_baseline
from functools import partial
from numpy.testing import assert_allclose
all_zero = [0]*EDGE_COUNT
single = all_zero.copy(); single[0] = 1
double = single.copy(); double[1] = 1
small_ds = experiment_splits['smoke']
with ledger.instrument(model), ledger.stage('stage_a_edge_equivalence', estimate=f'{SMOKE_COUNT} examples; baseline, one edge, two edges, official comparisons'):
    base = experiment.evaluate_state(all_zero,'smoke',fresh=True)
    official_base = evaluate_baseline(model,small_ds.to_dataloader(BATCH_SIZE),partial(experiment.metric,mean=False),quiet=True).numpy()
    assert_allclose(base,official_base,atol=3e-4,rtol=1e-4)
    for state in [single,double]:
        actual = experiment.evaluate_state(state,'smoke',fresh=True)
        official = experiment.official_binary_score(state,'smoke')
        assert_allclose(actual,official,atol=5e-4,rtol=1e-4)
        assert_allclose(actual,experiment.evaluate_state(state,'smoke',fresh=True),atol=3e-4,rtol=1e-4)
        assert_allclose(actual,experiment.evaluate_state(state,'smoke'),atol=0,rtol=0)
    assert_allclose(base,experiment.evaluate_state(all_zero,'smoke',fresh=True),atol=3e-4,rtol=1e-4)
    print('Endpoint and restoration checks passed. Baseline score:',base.mean(), 'one-edge score:',experiment.evaluate_state(single,'smoke').mean())

stage_a_edge_equivalence: 16 examples; baseline, one edge, two edges, official comparisons
Endpoint and restoration checks passed. Baseline score: 2.8993940949440002 one-edge score: -1.5862380862236023
stage_a_edge_equivalence: 192 actual forward batches, 13.65s wall


In [ ]:
# Synthetic HVP reference and a two-edge finite-difference check on real IOI examples.
H = torch.tensor([[2.,0.3],[0.3,-1.]],device='cuda',dtype=torch.float64)
q = lambda z: 0.5*z @ H @ z
point = torch.tensor([0.2,0.4],device='cuda',dtype=torch.float64)
vector = torch.tensor([1.,-0.7],device='cuda',dtype=torch.float64)
torch.testing.assert_close(hessian_vector_product(q,point,vector),H@vector,atol=1e-10,rtol=1e-10)
with ledger.instrument(model), ledger.stage('stage_a_hvp', estimate=f'2-edge directional HVP on {SMOKE_COUNT} official train examples'):
    z0 = torch.zeros(EDGE_COUNT,device='cuda',dtype=model.cfg.dtype)
    z0[:2] = 0.35
    direction = torch.zeros_like(z0); direction[:2] = torch.tensor([1.,-0.5],device='cuda')
    f = lambda z: experiment.continuous(z,'smoke').mean()
    hv = hessian_vector_product(f,z0,direction)
    eps = 0.01
    gradient = lambda z: torch.autograd.grad(f(z.requires_grad_(True)),z)[0].detach()
    finite = (gradient((z0+eps*direction).clone())-gradient((z0-eps*direction).clone()))/(2*eps)
    assert torch.isfinite(hv).all() and torch.isfinite(finite).all()
    assert_allclose(hv[:2].detach().cpu(),finite[:2].cpu(),atol=0.04,rtol=0.25)
    ledger.backward_calls += 4
    ledger.hvp_calls += 1
    print('HVP check passed:',hv[:2].detach().cpu().numpy(),'finite difference:',finite[:2].cpu().numpy())

stage_a_hvp: 2-edge directional HVP on 16 official train examples
HVP check passed: [-0.08418646  0.07741371] finite difference: [-0.08416176  0.07742643]
stage_a_hvp: 72 actual forward batches, 5.40s wall


In [ ]:
# A complete fixed-order restricted ACDC smoke trajectory; validation is not used for decisions.
print('Restricted ACDC smoke:',EDGE_COUNT,'edge trials,',SMOKE_COUNT,'train examples, ~',
      EDGE_COUNT*((SMOKE_COUNT+BATCH_SIZE-1)//BATCH_SIZE)*3,'forward batches before cache hits')
with ledger.instrument(model), ledger.stage('stage_a_exact_acdc', estimate=f'{EDGE_COUNT} edge trials on {SMOKE_COUNT} train examples'):
    final_state, rows = exact_acdc(experiment,'smoke',threshold=0.05)
trajectory = pd.DataFrame(rows)
trajectory.to_csv(OUT/'exact_acdc_trajectory.csv',index=False)
with ledger.instrument(model), ledger.stage('stage_a_final_validation', estimate='Held-out validation baseline and final-state scores'):
    validation_baseline = experiment.evaluate_state(all_zero,'validation').mean()
    validation_score = experiment.evaluate_state(final_state,'validation').mean()
    # The full official MIB evaluator must agree for the terminal binary mask too.
    assert_allclose(experiment.evaluate_state(final_state,'smoke'),
                    experiment.official_binary_score(final_state,'smoke'),atol=5e-4,rtol=1e-4)
print('Pruned:',sum(final_state),'of',EDGE_COUNT,'candidate edges; validation IOI baseline:',
      validation_baseline,'final:',validation_score,'change:',validation_score-validation_baseline)
pd.DataFrame(ledger.stages).to_csv(OUT/'runtime_breakdown.csv',index=False)
write_json(OUT/'stage_a_checks.json',{'edge_endpoint_equivalence':True,'hvp_synthetic_and_finite_difference':True,
    'restricted_acdc_completed':True,'smoke_train_examples':SMOKE_COUNT,
    'validation_baseline_score':float(validation_baseline),
    'validation_score':float(validation_score)})

Restricted ACDC smoke: 16 edge trials, 16 train examples, ~ 384 forward batches before cache hits
stage_a_exact_acdc: 16 edge trials on 16 train examples
stage_a_exact_acdc: 360 actual forward batches, 18.71s wall
stage_a_final_validation: Held-out validation baseline and final-state scores
stage_a_final_validation: 40 actual forward batches, 3.73s wall
Pruned: 10 of 16 candidate edges; validation IOI baseline: 2.440658926963806 final: 2.436887502670288 change: -0.0037714242935180664


## Stage B — H1 prediction experiment (opt in)

Run only after Stage A passes. At each state from several exact-ACDC trajectories, compare local gradient and randomized Hessian predictions against exact next-edge effects. Keep train examples for estimator construction and validation examples only for held-out assessment. A full rank-16 sketch requires 16 HVPs per expansion state and may cost more than exact edge trials.

In [ ]:
from src.sketch_acdc.research import collect_h1, rank_summary, calibrate_bound, hybrid_acdc
H1_THRESHOLDS = [0.05] if PILOT else [0.01, 0.05, 0.1]
SKETCH_RANKS = [2, 4] if PILOT else [2, 4, 8, 16]
H1_TRAIN_LIMIT = TRAIN_N//2 if PILOT else 16
if RUN_STAGE_B:
    estimated_states = EDGE_COUNT*len(H1_THRESHOLDS)
    estimated_hvps = estimated_states*sum(SKETCH_RANKS)
    print('Stage B estimate:',estimated_states,'states,',estimated_hvps,'HVP operations,',
          sum((EDGE_COUNT-i) for i in range(EDGE_COUNT))*len(H1_THRESHOLDS),
          'candidate effects on train and validation; this can be expensive on 12 GB.')
    with ledger.instrument(model), ledger.stage('stage_b_h1',estimate=f'{estimated_hvps} HVP operations plus exact candidate queries'):
        h1 = collect_h1(experiment,H1_THRESHOLDS,SKETCH_RANKS,H1_TRAIN_LIMIT,SEED,ledger)
    h1.to_csv(OUT/'edge_prediction_results.csv',index=False)
    ablation = rank_summary(h1)
    ablation.to_csv(OUT/'sketch_rank_ablation.csv',index=False)
    fig,axes=plt.subplots(1,3,figsize=(15,4))
    subset_h1=h1[h1['rank']==max(SKETCH_RANKS)]
    axes[0].scatter(subset_h1.true_validation_effect,subset_h1.first_order_prediction,s=8,label='first')
    axes[0].scatter(subset_h1.true_validation_effect,subset_h1.second_order_prediction,s=8,label='second')
    lo=min(subset_h1.true_validation_effect.min(),subset_h1.second_order_prediction.min())
    hi=max(subset_h1.true_validation_effect.max(),subset_h1.second_order_prediction.max())
    axes[0].plot([lo,hi],[lo,hi],'k--'); axes[0].set(xlabel='true effect',ylabel='predicted effect');axes[0].legend()
    for method,group in ablation.groupby('method'):
        axes[1].plot(group['rank'],group['mae'],marker='o',label=method)
    axes[1].set(xlabel='sketch rank',ylabel='held-out MAE');axes[1].legend()
    for method,column in [('first','first_order_decision_agrees'),('second','second_order_decision_agrees')]:
        axes[2].plot(subset_h1.groupby('removed_edges')[column].mean(),marker='o',label=method)
    axes[2].set(xlabel='removed edges',ylabel='decision agreement');axes[2].legend()
    fig.tight_layout();fig.savefig(OUT/'h1_predictions.png',dpi=150);plt.close(fig)
    pd.DataFrame(ledger.stages).to_csv(OUT/'runtime_breakdown.csv',index=False)
    print(ablation.to_string(index=False))
else:
    print('Stage B disabled; no H1 results have been measured.')

Stage B estimate: 16 states, 96 HVP operations, 136 candidate effects on train and validation; this can be expensive on 12 GB.
stage_b_h1: 96 HVP operations plus exact candidate queries
H1 threshold=0.05 state 1/16 pruned=0
H1 threshold=0.05 state 2/16 pruned=0
H1 threshold=0.05 state 3/16 pruned=0
H1 threshold=0.05 state 4/16 pruned=0
H1 threshold=0.05 state 5/16 pruned=0
H1 threshold=0.05 state 6/16 pruned=0
H1 threshold=0.05 state 7/16 pruned=0
H1 threshold=0.05 state 8/16 pruned=1
H1 threshold=0.05 state 9/16 pruned=2
H1 threshold=0.05 state 10/16 pruned=3
H1 threshold=0.05 state 11/16 pruned=4
H1 threshold=0.05 state 12/16 pruned=5
H1 threshold=0.05 state 13/16 pruned=6
H1 threshold=0.05 state 14/16 pruned=7
H1 threshold=0.05 state 15/16 pruned=8
H1 threshold=0.05 state 16/16 pruned=9
stage_b_h1: 7080 actual forward batches, 413.37s wall
 rank       method  n_decisions      mae  pearson  spearman  decision_agreement  false_prune_rate  false_retain_rate
    2  first_order          

## Stage C — H2 hybrid pruning (opt in)

This stage must follow a measured H1 benefit. Calibrate error intervals on held-out train queries, verify threshold-overlapping decisions exactly, compare refresh schedules and count every forward, backward and HVP. Restricted-graph IOI scores are not CPR/CMD.

In [ ]:
if RUN_STAGE_C:
    if not RUN_STAGE_B:
        raise RuntimeError('Run measured Stage B first; calibration uses train residuals from that stage.')
    C_THRESHOLD=0.05
    C_RANK=4
    C_REFRESH=[None,2] if PILOT else [None,2,4,'error',1]
    bound=calibrate_bound(h1[h1['rank']==C_RANK],quantile=0.9)
    first_bound=calibrate_bound(h1[h1['rank']==C_RANK],method='first_order',quantile=0.9)
    configurations=[('first_order_only',0,0,None,False),
                    ('first_order_verified',0,first_bound,2,True),
                    ('second_order_only',C_RANK,0,None,False),
                    ('second_order_verified',C_RANK,bound,2,True)]
    configurations += [('second_order_refresh_'+str(r),C_RANK,bound,r,True) for r in C_REFRESH]
    print('Stage C estimate:',len(configurations)*EDGE_COUNT,'decisions; refresh=1 costs about',
          EDGE_COUNT*C_RANK,'HVPs per algorithm plus exact verification and final validation.')
    quality=[]; decisions=[]
    with ledger.instrument(model),ledger.stage('stage_c_hybrid',estimate='Train-only decisions; held-out validation final quality'):
        for method,rank,error_bound,refresh,conservative in configurations:
            experiment.cache.clear()  # Compare cold compute, not another method's cached masks.
            start=time.perf_counter();before=len(ledger.calls);backwards=ledger.backward_calls
            state,detail,queries=hybrid_acdc(experiment,C_THRESHOLD,rank,H1_TRAIN_LIMIT,error_bound,
                                               refresh,conservative,SEED,ledger)
            detail['method']=method;decisions.append(detail)
            score=experiment.evaluate_state(state,'validation').mean()
            quality.append({'method':method,'refresh_after_removals':refresh,'validation_score':score,
                            'deviation_from_baseline':score-experiment.evaluate_state(all_zero,'validation').mean(),
                            'remaining_candidate_edges':EDGE_COUNT-sum(state),'exact_intervention_queries':queries,
                            'forward_batches':len(ledger.calls)-before,
                            'backward_calls':ledger.backward_calls-backwards,
                            'wall_seconds':time.perf_counter()-start})
        start=time.perf_counter();before=len(ledger.calls)
        experiment.cache.clear()
        exact_state,exact_rows=exact_acdc(experiment,'train',C_THRESHOLD)
        score=experiment.evaluate_state(exact_state,'validation').mean()
        quality.append({'method':'exact_acdc','refresh_after_removals':np.nan,'validation_score':score,
                        'deviation_from_baseline':score-experiment.evaluate_state(all_zero,'validation').mean(),
                        'remaining_candidate_edges':EDGE_COUNT-sum(exact_state),
                        'exact_intervention_queries':EDGE_COUNT,'forward_batches':len(ledger.calls)-before,
                        'backward_calls':0,'wall_seconds':time.perf_counter()-start})
    pd.concat(decisions).to_csv(OUT/'hybrid_acdc_results.csv',index=False)
    quality=pd.DataFrame(quality);quality.to_csv(OUT/'circuit_quality.csv',index=False)
    pd.DataFrame(ledger.stages).to_csv(OUT/'runtime_breakdown.csv',index=False)
    fig,axes=plt.subplots(1,3,figsize=(15,4))
    axes[0].scatter(quality.wall_seconds,quality.validation_score)
    axes[1].scatter(quality.forward_batches+quality.backward_calls,quality.validation_score)
    fractions=pd.concat(decisions).groupby('method').exact_verification.mean()
    hybrid_quality=quality.set_index('method').loc[fractions.index]
    axes[2].scatter(fractions,hybrid_quality.validation_score)
    axes[0].set(xlabel='wall seconds',ylabel='validation IOI score')
    axes[1].set(xlabel='forward + backward calls (unweighted)',ylabel='validation IOI score')
    axes[2].set(xlabel='fraction exactly verified',ylabel='validation IOI score')
    fig.tight_layout();fig.savefig(OUT/'h2_quality_cost.png',dpi=150);plt.close(fig)
    refresh_axis=quality.refresh_after_removals.map(lambda x:-1 if x=='error' else (0 if pd.isna(x) else x))
    fig,ax=plt.subplots();ax.scatter(refresh_axis,quality.wall_seconds,c=quality.validation_score)
    ax.set(xlabel='refresh interval (-1=train error trigger; 0=none/exact)',ylabel='wall seconds')
    fig.tight_layout();fig.savefig(OUT/'h2_refresh.png',dpi=150);plt.close(fig)
    print(quality.to_string(index=False))
else:
    print('Stage C disabled; no acceleration claim is made.')

Stage C estimate: 96 decisions; refresh=1 costs about 64 HVPs per algorithm plus exact verification and final validation.
stage_c_hybrid: Train-only decisions; held-out validation final quality
stage_c_hybrid: 3000 actual forward batches, 173.64s wall
                   method  refresh_after_removals  validation_score  deviation_from_baseline  remaining_candidate_edges  exact_intervention_queries  forward_batches  backward_calls  wall_seconds
         first_order_only                     NaN          2.436888                -0.003771                          6                           0               36               1      1.662190
     first_order_verified                     2.0          2.436888                -0.003771                          6                          16              492               5     24.168647
        second_order_only                     NaN          2.436888                -0.003771                          6                           0               8